# Notebook 32 — V6-F validation-only ensemble gate calibration

This T4 notebook repeats the exact three-seed V6-F gate recipe from Notebook 31. It changes no data, split, sampler, augmentation, stream architecture, or frozen temporal exercise classifier.

The only experiment is a predeclared ensemble threshold calibration. Candidate thresholds `0.30` through `0.80` in `0.01` increments are scored on validation only. A candidate must keep validation unknown recall at or above `0.85`; among eligible candidates, validation end-to-end Macro F1 is maximized, with supported accuracy and then the higher threshold used only as deterministic tie-breakers.

The selected threshold is frozen before test labels are scored. The reused 1,971-window internal test is reported only for the selected threshold and the `0.65` reference. It remains development evidence and cannot authorize production promotion.


In [ ]:
%pip install -q scikit-learn seaborn pandas==2.2.3


In [ ]:
from pathlib import Path
import hashlib
import json
import random
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from google.colab import files
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

RUN_SEEDS = [42, 137, 2026]
SEED = RUN_SEEDS[0]
FIXED_FUSION = "mean_probability"
REFERENCE_THRESHOLD = 0.65
THRESHOLD_GRID = np.round(np.arange(0.30, 0.801, 0.01), 2)
WINDOW = 64
STRIDE = 32
BATCH_SIZE = 64
MAX_EPOCHS = 40
PATIENCE = 8
MIN_VALIDATION_UNKNOWN_REJECTION = 0.85
GROUP_TEMPERATURE = 0.5  # frozen from V6-A.1
AUGMENTATION_CONFIG = {
    "horizontal_flip_probability": 0.5,
    "rotation_probability": 0.7,
    "max_rotation_degrees": 7.0,
    "speed_probability": 0.5,
    "speed_range": [0.85, 1.15],
    "frame_drop_probability": 0.35,
    "frame_drop_fraction_range": [0.05, 0.12],
    "coordinate_noise_probability": 0.8,
    "coordinate_noise_std": 0.008,
}
RETIRED_CLASSES = {"plank"}
NEW_TRAIN_GROUPS = {"youtube:Fk1wnNX4wQ8", "youtube:zxqCV_eQHZY"}
UNREVIEWED_GROUPS = {
    "youtube:oLsonHQJOUc",
    "youtube:c4TomZjzobY",
    "youtube:W2Bm9dfQjbk",
}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Experiment: V6-F validation-only ensemble gate calibration")
assert DEVICE.type == "cuda", "Select a T4 runtime before training."


## Download the exact V6-F evidence and Notebook 31 reference

Every training input remains checksum-pinned. Notebook 31's archived metrics are pinned separately so this run can verify deterministic reproduction before interpreting calibration.


In [ ]:
PINNED_COMMIT = "9874773e4681464bb00f5f0fb4dc3459caf75d6c"
REPOSITORY_RAW = (
    "https://raw.githubusercontent.com/bariscelikk1/cALorie/"
    f"{PINNED_COMMIT}/ml/artifacts"
)
LANDMARK_ARCHIVES = [
    ("v1/landmarks", "calorie_workoutfitness_landmarks_v1.zip"),
    ("v1/landmarks", "calorie_mmfit_landmarks_v1.zip"),
    ("v1/landmarks", "calorie_mmfit_unknown_v1.zip"),
    ("v2/landmarks", "calorie_youtube_mixed_landmarks_v3.zip"),
    ("v3/landmarks", "calorie_youtube_additional_landmarks_v1.zip"),
    ("v3/landmarks", "calorie_youtube_training_landmarks_v4.zip"),
    ("v5/landmarks", "calorie_workoutfitness_gate_hardneg_v1.zip"),
    ("youtube_long_v2/reviewed_clips", "calorie_youtube_long_workouts_v2_reviewed_clips.zip"),
]
EXPECTED_HASHES = {
    "calorie_workoutfitness_landmarks_v1.zip": "6dd4828e73f1ac1cd83a175de5bc1caf8465cbf61b72777467fba540af339aa9",
    "calorie_mmfit_landmarks_v1.zip": "2d3bff2c52842b276562e749fd7eaefe4925385efa7ff8c242c82ba1803413f1",
    "calorie_mmfit_unknown_v1.zip": "b0b113b3c5c7fc501b6b3ac4a7d2dade5fb4bccf3416d2eb85fa8b9197ae5596",
    "calorie_youtube_mixed_landmarks_v3.zip": "bd0426eda35654a95a8bac70b0357ba484e82bd29e6b3f71760fcc743f432cf4",
    "calorie_youtube_additional_landmarks_v1.zip": "0c60478086400fdfa6974afc6b091023b46e10783a71e40929aaa2b646d02743",
    "calorie_youtube_training_landmarks_v4.zip": "e0588c9a4a7d0257f710fec74b657e863cda5e0bc3285a640443863c9ede2b19",
    "calorie_workoutfitness_gate_hardneg_v1.zip": "b3aab2dbcefd709f1977c4d8d3f7d0326b50444ab44db4ba6b0cf64bbae0f4f6",
    "calorie_youtube_long_workouts_v2_reviewed_clips.zip": "84fb55b981820af702a94f0c0070960b24fdd1acb617097f1838c782cd942284",
    "hard_positive_windows.csv": "5a4b73c04ee85659570e7931ea0e84b35c0abeb16fd3f08238e629f6eca091c5",
    "split_assignments.csv": "8c013695fe095ceccc4ff65722d2a26e5a309674f5b877b840b71ba73dc085d6",
    "baseline_metrics.json": "baee2ea099402ca25b0e2bc322809dfa7a0f8d1e9969204792cd5d60bfb96b2b",
    "baseline_test_predictions.csv": "b34b6cada90355682ef7aa225e3d18dad0960dd79ce32856ebb524356734326c",
    "data_only_metrics.json": "cdc3d944063d12226a2ceaab6be9afd4e4cc23b6031dddc52a6ad86bb43c0670",
    "data_only_test_predictions.csv": "b379c30e563ea626d1f411c3ac372280820db03e7bd86dded647bc90a14adc65",
    "temporal_checkpoint.pt": "c30ecd625d77f31380a503f881eff5c540b8a7cdbbb71a39276c33b908947b92",
    "temporal_metrics.json": "27d15503aae2c23447294612569fcc3f96b08735ef3576f794c7db1d14824dcb",
    "temporal_test_predictions.csv": "7cab864936c31515ef65b2e55ea7a79ed282d2377468be7c68ca3b8120e76716",
}


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download(url, destination, expected=None):
    if destination.exists() and (expected is None or sha256(destination) == expected):
        print("Cached:", destination.name)
        return
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=300) as response, temporary.open("wb") as output:
        shutil.copyfileobj(response, output)
    if expected is not None:
        actual = sha256(temporary)
        assert actual == expected, f"SHA-256 mismatch for {destination.name}: {actual}"
    temporary.replace(destination)
    print(f"{destination.name}: {destination.stat().st_size / 1e6:.1f} MB")


zip_paths = []
for relative_root, name in LANDMARK_ARCHIVES:
    destination = Path("/content") / name
    download(
        f"{REPOSITORY_RAW}/{relative_root}/{name}", destination,
        EXPECTED_HASHES.get(name),
    )
    with zipfile.ZipFile(destination) as archive:
        assert archive.testzip() is None, f"Corrupt ZIP: {name}"
    zip_paths.append(destination)

HARD_POSITIVE_PATH = Path("/content/hard_positive_windows.csv")
download(
    f"{REPOSITORY_RAW}/v5/hard_positive_mining/hard_positive_windows.csv",
    HARD_POSITIVE_PATH, EXPECTED_HASHES["hard_positive_windows.csv"],
)
hard_positive_frame = pd.read_csv(HARD_POSITIVE_PATH)
assert hard_positive_frame.window_key.is_unique

V6_SPLIT_PATH = Path("/content/split_assignments.csv")
download(
    f"{REPOSITORY_RAW}/v6a1/model/split_assignments.csv",
    V6_SPLIT_PATH, EXPECTED_HASHES["split_assignments.csv"],
)
BASELINE_METRICS_PATH = Path("/content/baseline_metrics.json")
download(
    f"{REPOSITORY_RAW}/v6c/model/metrics.json",
    BASELINE_METRICS_PATH, EXPECTED_HASHES["baseline_metrics.json"],
)
BASELINE_PREDICTIONS_PATH = Path("/content/baseline_test_predictions.csv")
download(
    f"{REPOSITORY_RAW}/v6c/model/test_window_predictions.csv",
    BASELINE_PREDICTIONS_PATH, EXPECTED_HASHES["baseline_test_predictions.csv"],
)
baseline_metrics = json.loads(BASELINE_METRICS_PATH.read_text())
baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_PATH)
DATA_ONLY_METRICS_PATH = Path("/content/data_only_metrics.json")
download(f"{REPOSITORY_RAW}/v6c_longv2_data_retrain/model/metrics.json", DATA_ONLY_METRICS_PATH, EXPECTED_HASHES["data_only_metrics.json"])
DATA_ONLY_PREDICTIONS_PATH = Path("/content/data_only_test_predictions.csv")
download(f"{REPOSITORY_RAW}/v6c_longv2_data_retrain/model/test_window_predictions.csv", DATA_ONLY_PREDICTIONS_PATH, EXPECTED_HASHES["data_only_test_predictions.csv"])
data_only_metrics = json.loads(DATA_ONLY_METRICS_PATH.read_text())
data_only_predictions = pd.read_csv(DATA_ONLY_PREDICTIONS_PATH)
TEMPORAL_CHECKPOINT_PATH = Path("/content/temporal_checkpoint.pt")
download(f"{REPOSITORY_RAW}/v6e_complete_action_temporal/model/two_stage_tcn_v6e_temporal.pt", TEMPORAL_CHECKPOINT_PATH, EXPECTED_HASHES["temporal_checkpoint.pt"])
TEMPORAL_METRICS_PATH = Path("/content/temporal_metrics.json")
download(f"{REPOSITORY_RAW}/v6e_complete_action_temporal/model/metrics.json", TEMPORAL_METRICS_PATH, EXPECTED_HASHES["temporal_metrics.json"])
TEMPORAL_PREDICTIONS_PATH = Path("/content/temporal_test_predictions.csv")
download(f"{REPOSITORY_RAW}/v6e_complete_action_temporal/model/test_window_predictions.csv", TEMPORAL_PREDICTIONS_PATH, EXPECTED_HASHES["temporal_test_predictions.csv"])
temporal_metrics = json.loads(TEMPORAL_METRICS_PATH.read_text())
temporal_predictions = pd.read_csv(TEMPORAL_PREDICTIONS_PATH)
V6F_REFERENCE_PATH = Path("/content/v6f_reference_metrics.json")
download(
    "https://raw.githubusercontent.com/bariscelikk1/cALorie/e8f141e0861352cf9776d61289cbcc5112610c3b/"
    "ml/artifacts/v6f_multistream_gate/model/metrics.json",
    V6F_REFERENCE_PATH,
    "768f2950983d792ce7c9b1faa12d91578f2a8e37865fc387da7686814768ff22",
)
v6f_reference_metrics = json.loads(V6F_REFERENCE_PATH.read_text())
STABILITY_REFERENCE_PATH = Path("/content/v6f_seed_stability_metrics.json")
download(
    "https://raw.githubusercontent.com/bariscelikk1/cALorie/"
    "9520bcc810e03bb20f951ae707dde81ef090f73f/"
    "ml/artifacts/v6f_seed_stability/metrics.json",
    STABILITY_REFERENCE_PATH,
    "fa8c512984797756334bdf3c6b39bffced2d2afd58e2da9e424cae3f82a20f63",
)
stability_reference_metrics = json.loads(STABILITY_REFERENCE_PATH.read_text())

SOURCE_ROOT = Path("/content/v6c_longv2_sources")
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir()
for index, archive_path in enumerate(zip_paths):
    destination = SOURCE_ROOT / f"source_{index}"
    destination.mkdir()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination)

frames = []
for manifest_path in SOURCE_ROOT.rglob("manifest.csv"):
    frame = pd.read_csv(manifest_path).fillna("")
    frame["npz_path"] = frame["output_path"].map(
        lambda value: str(manifest_path.parent / value)
    )
    frames.append(frame)
assert frames, "No manifest.csv was found."
manifest = pd.concat(frames, ignore_index=True)
manifest = manifest[
    (manifest.status == "ok")
    & manifest.npz_path.map(lambda value: Path(value).exists())
].copy()
if "group_id" not in manifest or manifest.group_id.eq("").any():
    manifest["group_id"] = manifest.apply(
        lambda row: f"{row['source_dataset']}:{row['clip_id']}", axis=1
    )
assert NEW_TRAIN_GROUPS <= set(manifest.group_id)
print("Valid clips:", len(manifest), "Groups:", manifest.group_id.nunique())
display(manifest.canonical_class.value_counts().sort_index().rename("clips"))


## Preserve fixed windows and add train-only phase cycles

The old split and all evaluation windows remain unchanged. Supported training
clips receive at most six autocorrelation-period candidates. Each candidate is
resampled by cumulative joint motion, so slow and fast phases receive comparable
coverage without changing validation or test evidence.


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([
        np.interp(new_x, old_x, flat[:, index])
        for index in range(flat.shape[1])
    ], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


windows, original_labels, groups = [], [], []
clip_ids, sources, starts = [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    if sequence.ndim != 3 or sequence.shape[1:] != (17, 4):
        print("Skipping unexpected shape:", row.npz_path, sequence.shape)
        continue
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    clip_starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and clip_starts[-1] != len(sequence) - WINDOW:
        clip_starts.append(len(sequence) - WINDOW)
    for start in clip_starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([
                window, np.repeat(window[-1:], WINDOW - len(window), axis=0)
            ])
        windows.append(window)
        original_labels.append(row.canonical_class)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        sources.append(row.source_dataset)
        starts.append(start)

X = np.asarray(windows, dtype=np.float32)
original_labels = np.asarray(original_labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
sources = np.asarray(sources)
starts = np.asarray(starts, dtype=np.int64)
window_keys = np.asarray([
    f"{source}|{group}|{clip}|{start}"
    for source, group, clip, start in zip(sources, groups, clip_ids, starts)
])

# Reuse the exact group assignment exported by V6-A.1.
frozen_split = pd.read_csv(V6_SPLIT_PATH)
assert frozen_split.groupby("group_id")["split"].nunique().max() == 1
group_to_split = (
    frozen_split[["group_id", "split"]]
    .drop_duplicates("group_id")
    .set_index("group_id")["split"]
    .to_dict()
)
old_evidence_mask = np.asarray([group in group_to_split for group in groups])
assigned_split = np.asarray([group_to_split.get(group, "") for group in groups])
new_train_mask = np.isin(groups, list(NEW_TRAIN_GROUPS))
assigned_split[new_train_mask] = "train"
missing_groups = set(groups[assigned_split == ""])
assert missing_groups <= UNREVIEWED_GROUPS, (
    "Unexpected unresolved groups: " + str(sorted(missing_groups))
)
assert set(groups[new_train_mask]) == NEW_TRAIN_GROUPS
train_idx = np.flatnonzero(assigned_split == "train")
val_idx = np.flatnonzero(assigned_split == "validation")
test_idx = np.flatnonzero(assigned_split == "test")

labels = np.where(np.isin(original_labels, list(RETIRED_CLASSES)), "unknown", original_labels)
ALL_CLASSES = sorted(np.unique(labels).tolist())
SUPPORTED_CLASSES = [name for name in ALL_CLASSES if name != "unknown"]
assert len(SUPPORTED_CLASSES) == 14, SUPPORTED_CLASSES
all_to_id = {name: index for index, name in enumerate(ALL_CLASSES)}
supported_to_id = {name: index for index, name in enumerate(SUPPORTED_CLASSES)}
y_all = np.asarray([all_to_id[name] for name in labels], dtype=np.int64)
y_gate = (labels != "unknown").astype(np.int64)
y_exercise = np.asarray([supported_to_id.get(name, -1) for name in labels], dtype=np.int64)

assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert not (set(groups[val_idx]) & set(groups[test_idx]))
assert not (set(groups[train_idx]) & UNREVIEWED_GROUPS)
assert set(window_keys[old_evidence_mask]) == set(frozen_split.window_key), (
    "Reviewed window evidence changed since V6-A.1."
)
assert set(window_keys[test_idx]) == set(baseline_predictions.window_key)
assert len(test_idx) == 1971, len(test_idx)
assert set(window_keys[test_idx]) == set(data_only_predictions.window_key)
assert set(window_keys[test_idx]) == set(temporal_predictions.window_key)
baseline_truth_by_key = dict(zip(baseline_predictions.window_key, baseline_predictions.true_class))
assert all(baseline_truth_by_key[key] == truth for key, truth in zip(window_keys[test_idx], labels[test_idx])), "Frozen test labels changed relative to V6-C."
assert not set(groups[val_idx]) & NEW_TRAIN_GROUPS
assert not set(groups[test_idx]) & NEW_TRAIN_GROUPS
print("Excluded unresolved groups:", sorted(missing_groups))
print("New train-only groups:", sorted(NEW_TRAIN_GROUPS))

print("Windows:", X.shape)
print("Frozen groups:", len(group_to_split))
print("Active supported classes:", SUPPORTED_CLASSES)
print("Retired plank windows mapped to unknown:", int((original_labels == "plank").sum()))
for name, indices in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    print(name, "windows=", len(indices), "groups=", len(np.unique(groups[indices])))
    display(pd.Series(labels[indices]).value_counts().sort_index().rename("windows"))


new_indices = np.flatnonzero(new_train_mask)
data_delta_audit = pd.DataFrame({
    "canonical_class": labels[new_indices],
    "group_id": groups[new_indices],
    "window_key": window_keys[new_indices],
}).groupby(["canonical_class", "group_id"]).agg(
    windows=("window_key", "count")
).reset_index()
class_creator_support_matrix = pd.crosstab(
    pd.Series(labels[train_idx], name="canonical_class"),
    pd.Series(groups[train_idx], name="group_id"),
)
print("Added training windows:", len(new_indices))
display(data_delta_audit)
display(class_creator_support_matrix)


# Add train-only complete-cycle candidates while retaining every fixed window.
def estimate_period(sequence, minimum=12, maximum=90):
    xy = sequence[:, :, :2]
    energy = np.r_[0.0, np.linalg.norm(np.diff(xy, axis=0), axis=2).mean(axis=1)]
    energy = np.convolve(energy, np.ones(5) / 5.0, mode="same")
    centered = energy - energy.mean()
    upper = min(maximum, len(sequence) // 2)
    if upper <= minimum:
        return max(minimum, len(sequence))
    scores = []
    for lag in range(minimum, upper + 1):
        left, right = centered[:-lag], centered[lag:]
        denom = np.linalg.norm(left) * np.linalg.norm(right)
        scores.append(float(np.dot(left, right) / max(denom, 1e-8)))
    return minimum + int(np.argmax(scores))


def phase_resample(segment, target_length=WINDOW):
    xy = segment[:, :, :2]
    step_motion = np.r_[0.0, np.linalg.norm(np.diff(xy, axis=0), axis=2).mean(axis=1)]
    progress = np.cumsum(step_motion + 1e-5)
    progress = (progress - progress[0]) / max(progress[-1] - progress[0], 1e-8)
    targets = np.linspace(0.0, 1.0, target_length)
    flat = segment.reshape(len(segment), -1)
    sampled = np.stack([np.interp(targets, progress, flat[:, column]) for column in range(flat.shape[1])], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


temporal_windows, temporal_labels, temporal_groups = [], [], []
temporal_clip_ids, temporal_sources, temporal_starts, temporal_rows = [], [], [], []
for row in manifest.itertuples(index=False):
    canonical = "unknown" if row.canonical_class in RETIRED_CLASSES else row.canonical_class
    if (group_to_split.get(row.group_id) != "train" and row.group_id not in NEW_TRAIN_GROUPS) or canonical == "unknown":
        continue
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    if len(sequence) < 24:
        continue
    period = estimate_period(sequence)
    candidate_starts = list(range(0, max(1, len(sequence) - period + 1), period))
    if len(candidate_starts) > 6:
        candidate_starts = [candidate_starts[i] for i in np.linspace(0, len(candidate_starts) - 1, 6, dtype=int)]
    for cycle_index, start in enumerate(candidate_starts):
        end = min(len(sequence), start + period)
        if end - start < 12:
            continue
        temporal_windows.append(phase_resample(sequence[start:end]))
        temporal_labels.append(canonical); temporal_groups.append(row.group_id)
        temporal_clip_ids.append(row.clip_id); temporal_sources.append(row.source_dataset); temporal_starts.append(start)
        temporal_rows.append({"window_key": f"phase64|{row.source_dataset}|{row.group_id}|{row.clip_id}|{cycle_index}", "clip_id": row.clip_id, "group_id": row.group_id, "canonical_class": canonical, "source_start_frame": start, "source_end_frame": end, "estimated_period_frames": period, "source_frames": end - start})

assert temporal_windows, "No complete-action candidates were built."
temporal_frame = pd.DataFrame(temporal_rows)
base_count = len(X)
X = np.concatenate([X, np.asarray(temporal_windows, dtype=np.float32)])
original_labels = np.concatenate([original_labels, np.asarray(temporal_labels)])
labels = np.concatenate([labels, np.asarray(temporal_labels)])
groups = np.concatenate([groups, np.asarray(temporal_groups)])
clip_ids = np.concatenate([clip_ids, np.asarray(temporal_clip_ids)])
sources = np.concatenate([sources, np.asarray(temporal_sources)])
starts = np.concatenate([starts, np.asarray(temporal_starts, dtype=np.int64)])
window_keys = np.concatenate([window_keys, temporal_frame.window_key.to_numpy()])
assigned_split = np.concatenate([assigned_split, np.repeat("train", len(temporal_windows))])
y_all = np.asarray([all_to_id[name] for name in labels], dtype=np.int64)
y_gate = (labels != "unknown").astype(np.int64)
y_exercise = np.asarray([supported_to_id.get(name, -1) for name in labels], dtype=np.int64)
train_idx = np.flatnonzero(assigned_split == "train")
val_idx = np.flatnonzero(assigned_split == "validation")
test_idx = np.flatnonzero(assigned_split == "test")
temporal_mask = np.arange(len(X)) >= base_count
assert temporal_mask[val_idx].sum() == 0 and temporal_mask[test_idx].sum() == 0
assert set(window_keys[test_idx]) == set(data_only_predictions.window_key)
assert set(window_keys[test_idx]) == set(temporal_predictions.window_key)
print("Added phase-resampled training cycles:", int(temporal_mask.sum()))
display(temporal_frame.groupby("canonical_class").agg(cycles=("window_key", "count"), clips=("clip_id", "nunique"), median_period=("estimated_period_frames", "median")).round(2))


## Tempered hierarchical sampling and hard-positive weighting

Sampling first assigns equal probability to each class. Inside a class, each
source group receives probability proportional to `window_count ** 0.5`. This is
the midpoint between failed V6-A equal-group sampling (`** 0`) and raw window
sampling (`** 1`): long videos are restrained without making a one-window group
as influential as a large group. Notebook 13 hard-positive modifiers are
normalized inside the same class/group, so they change which windows are seen
without changing that group's total probability.

For the gate, “class” means supported or unknown. For the exercise model it
means one of the 14 active movements. `WeightedRandomSampler` then draws the
requested class -> group -> window distribution.


In [ ]:
hard_weight_by_key = dict(zip(
    hard_positive_frame.window_key,
    hard_positive_frame.recommended_sample_weight,
))
hard_positive_weights = np.asarray([
    float(hard_weight_by_key.get(key, 1.0)) for key in window_keys
], dtype=np.float64)
# Retired plank is now unknown and cannot receive a supported hard-positive boost.
hard_positive_weights[labels == "unknown"] = 1.0
matched_training_keys = (
    set(window_keys[train_idx][labels[train_idx] != "unknown"])
    & set(hard_weight_by_key)
)
assert not (set(window_keys[val_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect validation."
)
assert not (set(window_keys[test_idx]) & set(hard_weight_by_key)), (
    "Notebook 13 weights unexpectedly intersect test."
)
print("Notebook 13 keys:", len(hard_weight_by_key))
print("Matched active training keys:", len(matched_training_keys))
print("Upweighted active training windows:", int((hard_positive_weights[train_idx] > 1).sum()))


def hierarchical_window_weights(indices, hierarchy_labels, extra_weights=None):
    table = pd.DataFrame({
        "index": indices,
        "hierarchy_label": hierarchy_labels[indices],
        "group": groups[indices],
    })
    class_count = table["hierarchy_label"].nunique()
    group_sizes = table.groupby(
        ["hierarchy_label", "group"]
    )["index"].transform("count").astype(float)
    table["group_mass"] = group_sizes.pow(GROUP_TEMPERATURE)
    group_table = table[["hierarchy_label", "group"]].copy()
    group_table["group_mass"] = table["group_mass"]
    class_group_mass = (
        group_table.drop_duplicates(["hierarchy_label", "group"])
        .groupby("hierarchy_label")["group_mass"].sum()
    )
    table["extra_weight"] = (
        1.0 if extra_weights is None else extra_weights[table["index"].to_numpy()]
    )
    table["pair_extra_total"] = table.groupby(
        ["hierarchy_label", "group"]
    )["extra_weight"].transform("sum")
    weights = []
    for row in table.itertuples(index=False):
        weight = (
            1.0 / class_count
            * row.group_mass / class_group_mass.loc[row.hierarchy_label]
            * row.extra_weight / row.pair_extra_total
        )
        weights.append(weight)
    weights = np.asarray(weights, dtype=np.float64)
    return weights / weights.mean()


# Use strings here so the audit table remains human-readable.
gate_hierarchy = np.where(y_gate == 1, "supported", "unknown")
gate_train_weights = hierarchical_window_weights(
    train_idx, gate_hierarchy, hard_positive_weights
)
exercise_train_idx = train_idx[y_exercise[train_idx] >= 0]
exercise_val_idx = val_idx[y_exercise[val_idx] >= 0]
exercise_test_idx = test_idx[y_exercise[test_idx] >= 0]
exercise_train_weights = hierarchical_window_weights(
    exercise_train_idx, labels
)

sampler_audit = pd.DataFrame({
    "window_key": window_keys[train_idx],
    "canonical_class": labels[train_idx],
    "original_class": original_labels[train_idx],
    "group_id": groups[train_idx],
    "gate_target": gate_hierarchy[train_idx],
    "hard_positive_weight": hard_positive_weights[train_idx],
    "gate_sampler_weight": gate_train_weights,
})
exercise_weight_map = dict(zip(
    window_keys[exercise_train_idx], exercise_train_weights
))
sampler_audit["exercise_sampler_weight"] = sampler_audit.window_key.map(
    exercise_weight_map
).fillna(0.0)

display(sampler_audit.groupby("gate_target").agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_sampler_weight=("gate_sampler_weight", "sum"),
).round(3))
display(sampler_audit[sampler_audit.canonical_class != "unknown"].groupby(
    "canonical_class"
).agg(
    windows=("window_key", "count"),
    groups=("group_id", "nunique"),
    total_exercise_weight=("exercise_sampler_weight", "sum"),
).round(3))


In [ ]:
LEFT_RIGHT_PAIRS = [
    (1, 2), (3, 4), (5, 6), (7, 8),
    (9, 10), (11, 12), (13, 14), (15, 16),
]


def augment_pose_window(sample, force_all=False):
    sample = sample.copy()
    config = AUGMENTATION_CONFIG

    if force_all or np.random.random() < config["horizontal_flip_probability"]:
        sample[:, :, 0] *= -1.0
        for left, right in LEFT_RIGHT_PAIRS:
            sample[:, [left, right], :] = sample[:, [right, left], :]

    if force_all or np.random.random() < config["rotation_probability"]:
        degrees = np.random.uniform(
            -config["max_rotation_degrees"], config["max_rotation_degrees"]
        )
        radians = np.deg2rad(degrees)
        cosine, sine = np.cos(radians), np.sin(radians)
        x = sample[:, :, 0].copy()
        y = sample[:, :, 1].copy()
        sample[:, :, 0] = cosine * x - sine * y
        sample[:, :, 1] = sine * x + cosine * y

    if force_all or np.random.random() < config["speed_probability"]:
        speed = np.random.uniform(*config["speed_range"])
        center = (WINDOW - 1) / 2.0
        positions = (np.arange(WINDOW) - center) * speed + center
        flat = sample.reshape(WINDOW, -1)
        flat = np.stack([
            np.interp(positions, np.arange(WINDOW), flat[:, column])
            for column in range(flat.shape[1])
        ], axis=1)
        sample = flat.reshape(WINDOW, 17, 4).astype(np.float32)

    if force_all or np.random.random() < config["frame_drop_probability"]:
        fraction = np.random.uniform(*config["frame_drop_fraction_range"])
        count = max(1, int(round(WINDOW * fraction)))
        dropped = np.random.choice(np.arange(1, WINDOW), size=count, replace=False)
        sample[dropped] = sample[dropped - 1]

    if force_all or np.random.random() < config["coordinate_noise_probability"]:
        sample[:, :, :2] += np.random.normal(
            0, config["coordinate_noise_std"], sample[:, :, :2].shape
        ).astype(np.float32)
    return sample.astype(np.float32)



BONE_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]
ANGLE_TRIPLETS = [
    (5, 7, 9), (6, 8, 10),       # elbows
    (7, 5, 11), (8, 6, 12),      # shoulders
    (5, 11, 13), (6, 12, 14),    # hips
    (11, 13, 15), (12, 14, 16),  # knees
]
FEATURE_CONFIG = {
    "coordinates_xy": 34,
    "velocity_xy": 34,
    "bone_vectors_xy": 24,
    "joint_angles": 8,
}
FEATURE_DIM = sum(FEATURE_CONFIG.values())


def pose_to_features(sample):
    """Convert one [time, 17, 4] normalized pose window to [time, 100]."""
    xy = sample[:, :, :2].astype(np.float32)

    velocity = np.zeros_like(xy)
    velocity[1:] = xy[1:] - xy[:-1]

    bones = np.stack([
        xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES
    ], axis=1)

    angles = []
    for first, center, last in ANGLE_TRIPLETS:
        first_vector = xy[:, first] - xy[:, center]
        last_vector = xy[:, last] - xy[:, center]
        denominator = (
            np.linalg.norm(first_vector, axis=1)
            * np.linalg.norm(last_vector, axis=1)
        )
        cosine = np.sum(first_vector * last_vector, axis=1) / np.maximum(
            denominator, 1e-6
        )
        angles.append(np.arccos(np.clip(cosine, -1.0, 1.0)) / np.pi)
    angles = np.stack(angles, axis=1).astype(np.float32)

    features = np.concatenate([
        xy.reshape(WINDOW, -1),
        velocity.reshape(WINDOW, -1),
        bones.reshape(WINDOW, -1),
        angles,
    ], axis=1).astype(np.float32)
    assert features.shape == (WINDOW, FEATURE_DIM), features.shape
    assert np.isfinite(features).all()
    return features


class PoseWindowDataset(Dataset):
    def __init__(self, indices, targets, training=False):
        self.indices = np.asarray(indices)
        self.targets = targets
        self.training = training

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        actual = self.indices[item]
        sample = X[actual].copy()
        if self.training:
            sample = augment_pose_window(sample)
        features = pose_to_features(sample).T
        return torch.from_numpy(features), torch.tensor(
            self.targets[actual], dtype=torch.long
        )


def make_loaders(targets, train_indices, val_indices, test_indices, train_weights, seed):
    generator = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(
        torch.as_tensor(train_weights, dtype=torch.double),
        len(train_weights), replacement=True, generator=generator,
    )
    return (
        DataLoader(
            PoseWindowDataset(train_indices, targets, True),
            batch_size=BATCH_SIZE, sampler=sampler,
        ),
        DataLoader(
            PoseWindowDataset(val_indices, targets),
            batch_size=BATCH_SIZE, shuffle=False,
        ),
        DataLoader(
            PoseWindowDataset(test_indices, targets),
            batch_size=BATCH_SIZE, shuffle=False,
        ),
    )


gate_loaders = make_loaders(
    y_gate, train_idx, val_idx, test_idx, gate_train_weights, SEED
)
exercise_loaders = make_loaders(
    y_exercise, exercise_train_idx, exercise_val_idx, exercise_test_idx,
    exercise_train_weights, SEED + 1,
)
print("Feature blocks:", FEATURE_CONFIG)
print("Input features per frame:", FEATURE_DIM)
print("Gate split sizes:", [len(loader.dataset) for loader in gate_loaders])
print("Exercise split sizes:", [len(loader.dataset) for loader in exercise_loaders])


STREAMS = {"joint": 34, "bone": 24, "velocity": 34}
def pose_to_stream(sample, stream):
    xy = sample[:, :, :2].astype(np.float32)
    if stream == "joint": return xy.reshape(WINDOW, -1)
    if stream == "velocity":
        velocity = np.zeros_like(xy); velocity[1:] = xy[1:] - xy[:-1]
        return velocity.reshape(WINDOW, -1)
    if stream == "bone":
        return np.stack([xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES], axis=1).reshape(WINDOW, -1)
    raise ValueError(stream)

class StreamDataset(Dataset):
    def __init__(self, indices, training, stream): self.indices=np.asarray(indices); self.training=training; self.stream=stream
    def __len__(self): return len(self.indices)
    def __getitem__(self,item):
        actual=self.indices[item]; sample=X[actual].copy()
        if self.training: sample=augment_pose_window(sample)
        return torch.from_numpy(pose_to_stream(sample,self.stream).T), torch.tensor(y_gate[actual],dtype=torch.long)

def make_stream_loaders(stream, seed):
    generator=torch.Generator().manual_seed(seed)
    sampler=WeightedRandomSampler(torch.as_tensor(gate_train_weights,dtype=torch.double),len(gate_train_weights),replacement=True,generator=generator)
    return (DataLoader(StreamDataset(train_idx,True,stream),batch_size=BATCH_SIZE,sampler=sampler),DataLoader(StreamDataset(val_idx,False,stream),batch_size=BATCH_SIZE,shuffle=False),DataLoader(StreamDataset(test_idx,False,stream),batch_size=BATCH_SIZE,shuffle=False))


## Visual audit of a phase-resampled cycle

The preview uses a real temporal candidate before and after the frozen V6-C
augmentation. It is exported with the report.


In [ ]:
SKELETON_EDGES = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]
preview_candidates = train_idx[(labels[train_idx] == "squat") & temporal_mask[train_idx]]
preview_index = int(preview_candidates[0] if len(preview_candidates) else train_idx[0])
original_preview = X[preview_index].copy()
random_state = np.random.get_state()
np.random.seed(SEED + 1000)
augmented_preview = augment_pose_window(original_preview, force_all=True)
np.random.set_state(random_state)

fig, axes = plt.subplots(2, 6, figsize=(15, 6))
frame_indices = np.linspace(0, WINDOW - 1, 6, dtype=int)
for row, (title, sequence) in enumerate([
    ("original", original_preview), ("augmented", augmented_preview)
]):
    for axis, frame_index in zip(axes[row], frame_indices):
        xy = sequence[frame_index, :, :2]
        for left, right in SKELETON_EDGES:
            axis.plot(
                [xy[left, 0], xy[right, 0]],
                [-xy[left, 1], -xy[right, 1]],
                color="tab:blue", linewidth=1.4,
            )
        axis.scatter(xy[:, 0], -xy[:, 1], s=12, color="tab:red")
        axis.set_aspect("equal")
        axis.axis("off")
        axis.set_title(f"{title} t={frame_index}")
fig.suptitle(f"Notebook 32 phase-resampled cycle audit | {labels[preview_index]} | {window_keys[preview_index]}")
fig.tight_layout()
AUGMENTATION_PREVIEW_PATH = Path("/content/phase_cycle_augmentation_preview.png")
fig.savefig(AUGMENTATION_PREVIEW_PATH, dpi=170, bbox_inches="tight")
plt.show()
print("Preview:", AUGMENTATION_PREVIEW_PATH)


## Frozen architecture, fixed three-seed ensemble, validation-only threshold selection


In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


checkpoint=torch.load(TEMPORAL_CHECKPOINT_PATH,map_location=DEVICE)
frozen_exercise_model=SmallTCN(FEATURE_DIM,len(SUPPORTED_CLASSES)).to(DEVICE)
frozen_exercise_model.load_state_dict(checkpoint["exercise_state_dict"])
frozen_exercise_model.eval()
for parameter in frozen_exercise_model.parameters(): parameter.requires_grad=False


In [ ]:

def train_gate(stream, loaders, seed):
    seed_everything(seed)
    model = SmallTCN(STREAMS[stream], 2).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=3, factor=0.5
    )
    best_path = Path(f"/content/best_v6f_stability_{seed}_{stream}.pt")
    best_f1, stale, history = -1.0, 0, []

    def epoch(loader, training):
        model.train(training)
        losses, truth, prediction = [], [], []
        with (torch.enable_grad() if training else torch.no_grad()):
            for features, targets in loader:
                features, targets = features.to(DEVICE), targets.to(DEVICE)
                if training:
                    optimizer.zero_grad(set_to_none=True)
                logits = model(features)
                loss = criterion(logits, targets)
                if training:
                    loss.backward()
                    optimizer.step()
                losses.append(loss.item() * len(targets))
                truth.extend(targets.cpu().numpy())
                prediction.extend(logits.argmax(1).cpu().numpy())
        return (
            sum(losses) / len(truth),
            f1_score(truth, prediction, average="macro", zero_division=0),
        )

    for epoch_number in range(1, MAX_EPOCHS + 1):
        train_loss, train_f1 = epoch(loaders[0], True)
        val_loss, val_f1 = epoch(loaders[1], False)
        scheduler.step(val_f1)
        history.append({
            "seed": seed,
            "stream": stream,
            "epoch": epoch_number,
            "train_loss": train_loss,
            "train_macro_f1": train_f1,
            "val_loss": val_loss,
            "val_macro_f1": val_f1,
        })
        print(seed, stream, epoch_number, round(train_f1, 3), round(val_f1, 3))
        if val_f1 > best_f1 + 1e-4:
            best_f1, stale = val_f1, 0
            torch.save(model.state_dict(), best_path)
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    return model, history, best_f1, best_path


## Fixed validation and test scoring

All runs use mean stream probability and threshold `0.65`. Validation cannot choose a fusion, threshold, or winning seed. Test metrics are reported for every seed and for the predeclared arithmetic mean of their supported probabilities.


In [ ]:

def infer_stream(model, indices, stream, batch_size=256):
    probabilities = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            batch = np.stack([
                pose_to_stream(X[index], stream)
                for index in indices[start:start + batch_size]
            ]).transpose(0, 2, 1)
            tensor = torch.from_numpy(batch).to(DEVICE)
            probabilities.append(model(tensor).softmax(1).cpu().numpy())
    return np.concatenate(probabilities)


def exercise_probabilities(indices, batch_size=256):
    probabilities = []
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            batch = np.stack([
                pose_to_features(X[index])
                for index in indices[start:start + batch_size]
            ]).transpose(0, 2, 1)
            tensor = torch.from_numpy(batch).to(DEVICE)
            probabilities.append(
                frozen_exercise_model(tensor).softmax(1).cpu().numpy()
            )
    return np.concatenate(probabilities)


def score_predictions(indices, supported_scores, exercise_prediction, threshold):
    gate_prediction = (supported_scores >= threshold).astype(np.int64)
    end_prediction = np.asarray([
        SUPPORTED_CLASSES[class_id] if score >= threshold else "unknown"
        for score, class_id in zip(supported_scores, exercise_prediction)
    ])
    truth = labels[indices]
    unknown = truth == "unknown"
    supported = ~unknown
    metrics = {
        "gate_accuracy": float(accuracy_score(y_gate[indices], gate_prediction)),
        "gate_balanced_accuracy": float(
            balanced_accuracy_score(y_gate[indices], gate_prediction)
        ),
        "gate_macro_f1": float(
            f1_score(y_gate[indices], gate_prediction, average="macro", zero_division=0)
        ),
        "end_to_end_accuracy": float(accuracy_score(truth, end_prediction)),
        "end_to_end_balanced_accuracy": float(
            balanced_accuracy_score(truth, end_prediction)
        ),
        "end_to_end_macro_f1": float(
            f1_score(
                truth, end_prediction, labels=ALL_CLASSES,
                average="macro", zero_division=0,
            )
        ),
        "unknown_recall": float((end_prediction[unknown] == "unknown").mean()),
        "supported_accuracy": float(
            (end_prediction[supported] == truth[supported]).mean()
        ),
    }
    return gate_prediction, end_prediction, metrics


## Train reproducibly, select on validation, expose test once

The test probabilities may be computed while the models are resident on GPU, but test labels are not scored and no test metric is produced until validation has frozen one threshold. The test is never swept.


In [ ]:

REPORT_ROOT = Path("/content/calorie_v6f_validation_calibration_report")
if REPORT_ROOT.exists():
    shutil.rmtree(REPORT_ROOT)
REPORT_ROOT.mkdir()

exercise_val_probability = exercise_probabilities(val_idx)
exercise_test_probability = exercise_probabilities(test_idx)
exercise_val_prediction = exercise_val_probability.argmax(1)
exercise_test_prediction = exercise_test_probability.argmax(1)

baseline_prediction_by_key = dict(zip(
    baseline_predictions.window_key, baseline_predictions.predicted_class
))
stream_rows = []
history_rows = []
val_scores_by_seed = []
test_scores_by_seed = []

for run_seed in RUN_SEEDS:
    print("\n=== Seed", run_seed, "===")
    stream_results = {}
    for offset, stream in enumerate(STREAMS):
        training_seed = run_seed + 100 + offset
        loaders = make_stream_loaders(stream, training_seed)
        model, history, best_f1, best_path = train_gate(
            stream, loaders, training_seed
        )
        stream_results[stream] = {
            "model": model,
            "best_validation_f1": best_f1,
            "best_path": best_path,
            "training_seed": training_seed,
        }
        history_rows.extend(history)
        stream_rows.append({
            "run_seed": run_seed,
            "training_seed": training_seed,
            "stream": stream,
            "best_validation_macro_f1": best_f1,
        })

    val_stream_probability = {
        stream: infer_stream(result["model"], val_idx, stream)[:, 1]
        for stream, result in stream_results.items()
    }
    test_stream_probability = {
        stream: infer_stream(result["model"], test_idx, stream)[:, 1]
        for stream, result in stream_results.items()
    }
    val_score = np.stack(
        [val_stream_probability[stream] for stream in STREAMS], axis=1
    ).mean(axis=1)
    test_score = np.stack(
        [test_stream_probability[stream] for stream in STREAMS], axis=1
    ).mean(axis=1)
    val_scores_by_seed.append(val_score)
    test_scores_by_seed.append(test_score)

    for stream, result in stream_results.items():
        torch.save({
            "state_dict": result["model"].state_dict(),
            "stream": stream,
            "input_features": STREAMS[stream],
            "run_seed": run_seed,
            "training_seed": result["training_seed"],
            "fusion": FIXED_FUSION,
            "calibration_protocol": "validation_only_after_mean_seed_fusion",
        }, REPORT_ROOT / f"seed_{run_seed}_{stream}_gate.pt")
        del result["model"]
    del stream_results
    torch.cuda.empty_cache()

ensemble_val_score = np.mean(np.stack(val_scores_by_seed), axis=0)
ensemble_test_score = np.mean(np.stack(test_scores_by_seed), axis=0)

# Threshold selection is validation-only. No test metric exists above this line.
threshold_rows = []
for threshold in THRESHOLD_GRID:
    _, _, validation_metrics = score_predictions(
        val_idx, ensemble_val_score, exercise_val_prediction, float(threshold)
    )
    threshold_rows.append({
        "threshold": float(threshold),
        **{f"validation_{key}": value for key, value in validation_metrics.items()},
        "eligible_unknown_recall_floor": bool(
            validation_metrics["unknown_recall"]
            >= MIN_VALIDATION_UNKNOWN_REJECTION
        ),
    })
validation_threshold_sweep = pd.DataFrame(threshold_rows)
eligible = validation_threshold_sweep[
    validation_threshold_sweep.eligible_unknown_recall_floor
].copy()
assert len(eligible), "No validation threshold satisfies the unknown-recall floor."
selected_row = eligible.sort_values(
    [
        "validation_end_to_end_macro_f1",
        "validation_supported_accuracy",
        "threshold",
    ],
    ascending=[False, False, False],
).iloc[0]
SELECTED_THRESHOLD = float(selected_row.threshold)
selection_frozen = True
print("Validation-selected threshold:", SELECTED_THRESHOLD)

# Test labels are scored only after validation freezes SELECTED_THRESHOLD.
_, reference_val_prediction, reference_val_metrics = score_predictions(
    val_idx, ensemble_val_score, exercise_val_prediction, REFERENCE_THRESHOLD
)
_, selected_val_prediction, selected_val_metrics = score_predictions(
    val_idx, ensemble_val_score, exercise_val_prediction, SELECTED_THRESHOLD
)
_, reference_test_prediction, reference_test_metrics = score_predictions(
    test_idx, ensemble_test_score, exercise_test_prediction, REFERENCE_THRESHOLD
)
_, selected_test_prediction, selected_test_metrics = score_predictions(
    test_idx, ensemble_test_score, exercise_test_prediction, SELECTED_THRESHOLD
)

comparison_rows = []
for variant, threshold, validation_metrics, test_metrics in [
    (
        "reference_0_65", REFERENCE_THRESHOLD,
        reference_val_metrics, reference_test_metrics,
    ),
    (
        "validation_selected", SELECTED_THRESHOLD,
        selected_val_metrics, selected_test_metrics,
    ),
]:
    comparison_rows.append({
        "variant": variant,
        "threshold": threshold,
        **{f"validation_{key}": value for key, value in validation_metrics.items()},
        **{f"test_{key}": value for key, value in test_metrics.items()},
    })
selected_vs_reference = pd.DataFrame(comparison_rows)

seed_metric_rows = []
for run_seed, val_score, test_score in zip(
    RUN_SEEDS, val_scores_by_seed, test_scores_by_seed
):
    _, _, val_metrics = score_predictions(
        val_idx, val_score, exercise_val_prediction, SELECTED_THRESHOLD
    )
    _, _, test_metrics = score_predictions(
        test_idx, test_score, exercise_test_prediction, SELECTED_THRESHOLD
    )
    seed_metric_rows.append({
        "run_seed": run_seed,
        "selected_threshold": SELECTED_THRESHOLD,
        **{f"validation_{key}": value for key, value in val_metrics.items()},
        **{f"test_{key}": value for key, value in test_metrics.items()},
    })
seed_metrics = pd.DataFrame(seed_metric_rows)

class_rows = []
for variant, prediction in [
    ("reference_0_65", reference_test_prediction),
    ("validation_selected", selected_test_prediction),
]:
    report = classification_report(
        labels[test_idx], prediction, labels=ALL_CLASSES,
        output_dict=True, zero_division=0,
    )
    for class_name in ALL_CLASSES:
        class_rows.append({
            "variant": variant,
            "class_name": class_name,
            **report[class_name],
        })
class_metrics = pd.DataFrame(class_rows)

creator_rows = []
for group_id in sorted(set(groups[test_idx])):
    mask = groups[test_idx] == group_id
    truth = labels[test_idx][mask]
    present = sorted(set(truth))
    baseline_group_prediction = np.asarray([
        baseline_prediction_by_key[key]
        for key in window_keys[test_idx][mask]
    ])
    reference_group_prediction = reference_test_prediction[mask]
    selected_group_prediction = selected_test_prediction[mask]
    baseline_f1 = f1_score(
        truth, baseline_group_prediction, labels=present,
        average="macro", zero_division=0,
    )
    reference_f1 = f1_score(
        truth, reference_group_prediction, labels=present,
        average="macro", zero_division=0,
    )
    selected_f1 = f1_score(
        truth, selected_group_prediction, labels=present,
        average="macro", zero_division=0,
    )
    creator_rows.append({
        "group_id": group_id,
        "windows": int(mask.sum()),
        "v6c_accuracy": float(accuracy_score(truth, baseline_group_prediction)),
        "reference_accuracy": float(accuracy_score(truth, reference_group_prediction)),
        "selected_accuracy": float(accuracy_score(truth, selected_group_prediction)),
        "v6c_present_macro_f1": float(baseline_f1),
        "reference_present_macro_f1": float(reference_f1),
        "selected_present_macro_f1": float(selected_f1),
        "selected_accuracy_delta_vs_v6c": float(
            accuracy_score(truth, selected_group_prediction)
            - accuracy_score(truth, baseline_group_prediction)
        ),
        "selected_present_macro_f1_delta_vs_v6c": float(
            selected_f1 - baseline_f1
        ),
        "selected_present_macro_f1_delta_vs_reference": float(
            selected_f1 - reference_f1
        ),
    })
creator_metrics = pd.DataFrame(creator_rows)

reference_notebook31 = stability_reference_metrics["predeclared_ensemble"]
assert abs(
    reference_test_metrics["end_to_end_macro_f1"]
    - reference_notebook31["end_to_end_macro_f1"]
) <= 1e-6, "Notebook 31 deterministic reference was not reproduced."
assert abs(
    reference_test_metrics["unknown_recall"]
    - reference_notebook31["unknown_recall"]
) <= 1e-6, "Notebook 31 unknown recall was not reproduced."

creator_index = creator_metrics.set_index("group_id")
youtube_row = creator_index.loc["youtube:fegQSC-nUtg"]
w13_row = creator_index.loc["mm-fit:w13"]
w18_row = creator_index.loc["mm-fit:w18"]
outcome_gates = {
    "selection_used_validation_only": True,
    "selected_validation_unknown_recall_at_least_0_85": bool(
        selected_val_metrics["unknown_recall"]
        >= MIN_VALIDATION_UNKNOWN_REJECTION
    ),
    "notebook31_reference_reproduced": True,
    "selected_test_macro_f1_not_below_reference": bool(
        selected_test_metrics["end_to_end_macro_f1"]
        >= reference_test_metrics["end_to_end_macro_f1"]
    ),
    "youtube_present_f1_delta_vs_v6c_at_least_minus_0_05": bool(
        youtube_row.selected_present_macro_f1_delta_vs_v6c >= -0.05
    ),
    "w13_present_f1_delta_vs_v6c_at_least_minus_0_05": bool(
        w13_row.selected_present_macro_f1_delta_vs_v6c >= -0.05
    ),
    "w18_present_f1_delta_vs_v6c_at_least_minus_0_05": bool(
        w18_row.selected_present_macro_f1_delta_vs_v6c >= -0.05
    ),
}
outcome_gates["research_calibration_pass"] = bool(all(outcome_gates.values()))
outcome_gates["production_promotion_allowed"] = False

metrics = {
    "experiment": "v6f_validation_only_ensemble_gate_calibration",
    "pinned_training_source_commit": PINNED_COMMIT,
    "notebook31_reference_commit": "9520bcc810e03bb20f951ae707dde81ef090f73f",
    "run_seeds": RUN_SEEDS,
    "fusion": FIXED_FUSION,
    "reference_threshold": REFERENCE_THRESHOLD,
    "threshold_grid": [float(value) for value in THRESHOLD_GRID],
    "validation_unknown_recall_floor": MIN_VALIDATION_UNKNOWN_REJECTION,
    "selection_objective": (
        "maximize validation end-to-end Macro F1; tie-break by validation "
        "supported accuracy, then higher threshold"
    ),
    "selected_threshold": SELECTED_THRESHOLD,
    "selection_frozen_before_test_scoring": selection_frozen,
    "test_thresholds_scored": [REFERENCE_THRESHOLD, SELECTED_THRESHOLD],
    "test_window_count": int(len(test_idx)),
    "reference_metrics": reference_test_metrics,
    "selected_metrics": selected_test_metrics,
    "seed_summary_selected_threshold": seed_metrics.to_dict(orient="records"),
    "outcome_gates": outcome_gates,
    "test_used_for_threshold_selection": False,
    "internal_test_is_reused_development_evidence": True,
    "production_promoted": False,
}

(REPORT_ROOT / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
validation_threshold_sweep.to_csv(
    REPORT_ROOT / "validation_threshold_sweep.csv", index=False
)
selected_vs_reference.to_csv(
    REPORT_ROOT / "selected_vs_reference_metrics.csv", index=False
)
seed_metrics.to_csv(
    REPORT_ROOT / "seed_metrics_selected_threshold.csv", index=False
)
pd.DataFrame(stream_rows).to_csv(
    REPORT_ROOT / "seed_stream_validation.csv", index=False
)
pd.DataFrame(history_rows).to_csv(
    REPORT_ROOT / "training_history.csv", index=False
)
class_metrics.to_csv(REPORT_ROOT / "class_metrics.csv", index=False)
creator_metrics.to_csv(REPORT_ROOT / "creator_metrics.csv", index=False)
pd.DataFrame({
    "window_key": window_keys[test_idx],
    "group_id": groups[test_idx],
    "true_class": labels[test_idx],
    "ensemble_supported_score": ensemble_test_score,
    "exercise_prediction": [
        SUPPORTED_CLASSES[class_id] for class_id in exercise_test_prediction
    ],
    "reference_prediction": reference_test_prediction,
    "selected_prediction": selected_test_prediction,
}).to_csv(REPORT_ROOT / "ensemble_test_predictions.csv", index=False)

matrix = confusion_matrix(
    labels[test_idx], selected_test_prediction, labels=ALL_CLASSES
)
plt.figure(figsize=(12, 10))
sns.heatmap(
    matrix, annot=True, fmt="d", cmap="Blues",
    xticklabels=ALL_CLASSES, yticklabels=ALL_CLASSES,
)
plt.tight_layout()
plt.savefig(REPORT_ROOT / "selected_confusion_matrix.png", dpi=180)
plt.close()

plt.figure(figsize=(9, 5))
plt.plot(
    validation_threshold_sweep.threshold,
    validation_threshold_sweep.validation_end_to_end_macro_f1,
    label="validation end-to-end Macro F1",
)
plt.plot(
    validation_threshold_sweep.threshold,
    validation_threshold_sweep.validation_unknown_recall,
    label="validation unknown recall",
)
plt.axhline(
    MIN_VALIDATION_UNKNOWN_REJECTION, color="black", linestyle=":",
    label="unknown-recall floor",
)
plt.axvline(
    REFERENCE_THRESHOLD, color="tab:gray", linestyle="--",
    label="reference threshold",
)
plt.axvline(
    SELECTED_THRESHOLD, color="tab:red", linestyle="--",
    label="validation-selected threshold",
)
plt.ylim(0, 1)
plt.xlabel("gate threshold")
plt.ylabel("validation score")
plt.legend()
plt.tight_layout()
plt.savefig(REPORT_ROOT / "validation_calibration_curve.png", dpi=180)
plt.close()

sampler_audit.to_csv(REPORT_ROOT / "sampler_audit.csv", index=False)
temporal_frame.to_csv(REPORT_ROOT / "temporal_cycle_audit.csv", index=False)
data_delta_audit.to_csv(REPORT_ROOT / "data_delta_audit.csv", index=False)
class_creator_support_matrix.to_csv(
    REPORT_ROOT / "class_creator_support_matrix.csv"
)
pd.DataFrame({
    "window_key": window_keys,
    "group_id": groups,
    "active_class": labels,
    "split": assigned_split,
}).query('split != ""').to_csv(REPORT_ROOT / "split_assignments.csv", index=False)
shutil.copy2(
    TEMPORAL_CHECKPOINT_PATH,
    REPORT_ROOT / "frozen_temporal_classifier_checkpoint.pt",
)
if AUGMENTATION_PREVIEW_PATH.exists():
    shutil.copy2(
        AUGMENTATION_PREVIEW_PATH,
        REPORT_ROOT / AUGMENTATION_PREVIEW_PATH.name,
    )

source_paths = zip_paths + [
    HARD_POSITIVE_PATH, V6_SPLIT_PATH,
    BASELINE_METRICS_PATH, BASELINE_PREDICTIONS_PATH,
    DATA_ONLY_METRICS_PATH, DATA_ONLY_PREDICTIONS_PATH,
    TEMPORAL_CHECKPOINT_PATH, TEMPORAL_METRICS_PATH,
    TEMPORAL_PREDICTIONS_PATH, V6F_REFERENCE_PATH,
    STABILITY_REFERENCE_PATH,
]
(REPORT_ROOT / "source_hashes.json").write_text(json.dumps(
    {path.name: sha256(path) for path in source_paths}, indent=2
) + "\n")
checksum_lines = [
    f"{sha256(path)}  {path.relative_to(REPORT_ROOT)}"
    for path in sorted(
        item for item in REPORT_ROOT.rglob("*") if item.is_file()
    )
]
(REPORT_ROOT / "SHA256SUMS.txt").write_text(
    "\n".join(checksum_lines) + "\n"
)
zip_path = shutil.make_archive(
    "/content/calorie_v6f_validation_calibration_report",
    "zip", root_dir=REPORT_ROOT,
)
display(validation_threshold_sweep)
display(selected_vs_reference)
display(creator_metrics[
    creator_metrics.group_id.isin([
        "youtube:fegQSC-nUtg", "mm-fit:w13", "mm-fit:w18"
    ])
])
print(json.dumps(outcome_gates, indent=2))
print("Created:", zip_path, round(Path(zip_path).stat().st_size / 1e6, 1), "MB")
files.download(zip_path)


## Return the calibration report

Upload `calorie_v6f_validation_calibration_report.zip` back to the project. Keep the file unchanged; its internal checksum manifest covers every report artifact and checkpoint.
